In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from Bio import SeqIO
from tqdm import tqdm
import json
import os
import logging
import numpy as np
import matplotlib.patches as patches

# Import all our custom pipeline modules
from instanexus import preprocessing
from instanexus import assembly
from instanexus import visualization
from instanexus import helpers

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
pd.options.display.max_colwidth = None

In [ ]:
os.chdir('../../../../')

print(f"Current working directory: {os.getcwd()}")

In [ ]:
FIGURES_DIR = Path("figures")
JSON_DIR = Path("json")
print(FIGURES_DIR)

In [ ]:
# Path to the new raw data you want to test
INPUT_CSV = "inputs/bsa.csv"
METADATA_PATH = "json/sample_metadata.json"
CONTAMINANTS_PATH = "fasta/contaminants.fasta"
RUN_NAME = Path(INPUT_CSV).stem
REFERENCE_MODE = True

# Metadata params
#MASS_ERR_LIMIT = 20
#MAX_IRT_ERROR = 60
#MIN_ENTROPY = 1
#PROSIT_FILTER = True
#Z_SCORE_THRESHOLD = -0.5


# Assembly params
ASSEMBLY_MODE = "dbg_weighted"
CHAIN = ""
CONFIDENCE_THRESHOLD = 0.8
MIN_LENGTH = 7
MAX_LENGHT = 20
FDR_THRESHOLD = 0.1
KMER_SIZE = 7
MIN_OVERLAP = 3
SIZE_THRESHOLD = 10
MIN_IDENTITY = 0.8
MAX_MISMATCHES = 100

# Clustering params
MIN_SEQ_ID = 0.85
COVERAGE = 0.8

In [ ]:
#base_output_folder = Path(BASE_OUTPUT_FOLDER) / RUN_NAME

# Build the unique experiment folder name
folder_name_parts = [f"{ASSEMBLY_MODE}"]

if CONFIDENCE_THRESHOLD is not None:
    folder_name_parts.append(f"c{CONFIDENCE_THRESHOLD}")

if "dbg" in ASSEMBLY_MODE:
    folder_name_parts.append(f"ks{KMER_SIZE}")

folder_name_parts.append(f"mo{MIN_OVERLAP}")
folder_name_parts.append(f"ts{SIZE_THRESHOLD}")

if REFERENCE_MODE:
    folder_name_parts.extend([f"mi{MIN_IDENTITY}", f"mm{MAX_MISMATCHES}"])

run_folder_name = "_".join(folder_name_parts)
#experiment_folder = base_output_folder / run_folder_name


run_id_str = f"[{RUN_NAME} @ {run_folder_name}]"

logger.info(f"Pipeline starting for run: {run_id_str}")

In [ ]:
sample_metadata = preprocessing.get_sample_metadata(
    run=RUN_NAME, 
    chain=CHAIN, 
    json_path=METADATA_PATH
)

In [ ]:
proteases = sample_metadata["proteases"]
protein = sample_metadata["protein"]
protein_norm = preprocessing.normalize_sequence(protein)

In [ ]:
print(f"Sample uses proteases: {proteases}")
print(f"Protein sequence length: {len(protein)} amino acids")
print(f"Normalized protein sequence: {protein_norm}")

In [ ]:
original_data = pd.read_csv(INPUT_CSV)

In [ ]:
original_data

In [ ]:
cols_to_keep = [
    'experiment_name',
    'prediction_untokenised',
    'instanovo_token_log_probabilities',
    'calibrated_confidence',    
    'psm_q_value',
    'delta_mass_ppm',
    'Mass Error',               
    'is_missing_prosit_features', 
    'ion_match_intensity',
    'ion_matches',
    'iRT',
    'iRT error',
    'is_missing_irt_error',
    'predicted iRT',
    'margin',
    'entropy',
    'z-score'
    ]

data = original_data[cols_to_keep].copy()

In [ ]:
# show me the number of predition_untokenised with no values for calibrated_confidence
num_missing_calibrated_conf = data['calibrated_confidence'].isnull().sum()
print(f"Number of prediction_untokenised with no values for calibrated_confidence: {num_missing_calibrated_conf}")


In [ ]:
data.rename(columns={'calibrated_confidence': 'conf'}, inplace=True)

In [ ]:
data["protease"] = data["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)

protease_col = data.pop("protease")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "protease", protease_col)

In [ ]:
# show me the number of predition_untokenised with no values for conf
num_missing_conf = data['conf'].isnull().sum()
print(f"Number of prediction_untokenised with no values for conf: {num_missing_conf}")

In [ ]:
data

In [ ]:
data = data.dropna(subset=["prediction_untokenised"])

In [ ]:
data["cleaned_preds"] = data["prediction_untokenised"].apply(preprocessing.remove_modifications)

# move cleaned_preds next to prediction_untokenised
cleaned_preds_col = data.pop("cleaned_preds")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "cleaned_preds", cleaned_preds_col)

In [ ]:
cleaned_psms = data["cleaned_preds"].tolist()

In [ ]:
filtered_psms = preprocessing.filter_contaminants(
    cleaned_psms, RUN_NAME , CONTAMINANTS_PATH
)

In [ ]:
data = data[data["cleaned_preds"].isin(filtered_psms)]

In [ ]:
data.drop(columns=['prediction_untokenised'], inplace=True)

In [ ]:
#data.loc[:, "mapped"] = data["cleaned_preds"].apply(lambda x: x in protein_norm)

data["mapped"] = data["cleaned_preds"].apply(
    lambda x: "True" if x in protein_norm else "False"
)

In [ ]:
data = data[data['cleaned_preds'].str.len() >= MIN_LENGTH]


data = data[data['cleaned_preds'].str.len() <= MAX_LENGHT]

In [ ]:
# show me value counts of mapped vs unmapped
data['mapped'].value_counts()

In [ ]:
# remove the column "instanovo_token_log_probabilities"
data = data.drop(columns=["instanovo_token_log_probabilities"])

In [ ]:
data.head(3)

### Distribution of PSMs

In the following plot we show how every single PSM, colored by its protease, maps to the reference protein.

In [ ]:
visualization.plot_map_unmap_distribution(data, RUN_NAME, FIGURES_DIR, 0, 1, False, title=False)

### Ratios of mapped/unmapped
Considering the confidence and the fdr, we show the percentages of mapped and unmapped sequences of the PSM at specific ranges.

In [ ]:
visualization.plot_ratios_mapped_unmapped_threshold_set(RUN_NAME, data, reference=protein_norm, folder=FIGURES_DIR, ratio=1)

### Ratios of mapped/unmapped across all the FDR scores

In [ ]:
visualization.plot_fdr_mapped_unmapped_ratio_across_all_range(RUN_NAME, data, protein_norm, FIGURES_DIR)

### Lineplot of coverage at different level of FDR

In [ ]:
visualization.plot_coverage_vs_fdr_curve(RUN_NAME, data, protein_norm, FIGURES_DIR)

### Distribution of each proteases across calibrated confidence score

In [ ]:
visualization.plot_protease_confidence_ridges(data, colors_path='json/protease_colors.json', folder=FIGURES_DIR)

### Adding quantification data

In [ ]:
data_abundance = preprocessing.add_quantification_data(data, RUN_NAME, FDR_THRESHOLD)

## Pie chart sunberst

In [ ]:
visualization.plot_sunburst(data_abundance, FIGURES_DIR, f"fig2f_{RUN_NAME}_sunburst.svg", "json/protease_colors.json")

In [ ]:
sequences = data_abundance['cleaned_preds'].tolist()

In [ ]:
mapped_data = visualization.process_protein_contigs_scaffold(
    sequences,
    protein_norm,
    max_mismatches=10,
    min_identity=0.8
)

In [ ]:
visualization.mapping_psms_protease_associated(
    mapped_sequences=mapped_data,
    prot_seq=protein_norm,
    labels=data_abundance['protease'].tolist(),
    output_folder=FIGURES_DIR,
    output_file=f"fig2g_{RUN_NAME}_protease_mapping.svg",
    json_colors_path="json/protease_colors.json",
    show_figure=True
)

## Assemblers

In [ ]:
assembler = assembly.Assembler(
    mode="dbg_weighted",
    kmer_size=7,
    min_overlap=3,
    size_threshold=10,
    min_weight=2,
    refine_rounds=5
)

In [ ]:
assembler = assembly.Assembler(
    mode="greedy",
    min_overlap=3,
    size_threshold=0,
    min_weight=2,
    refine_rounds=5
)

In [ ]:
scaffolds = assembler.run(sequences=sequences, df_full=data_abundance)

In [ ]:
mapped_scaffolds = visualization.process_protein_contigs_scaffold(
    scaffolds, protein_norm, 10, 0.8)

In [ ]:
visualization.mapping_sequences(
    mapped_sequences=mapped_scaffolds,
    prot_seq=protein_norm,
    category="bsa",
    config_json_path=f"{JSON_DIR}/colors.json",
    output_folder=FIGURES_DIR,
    output_file=f"fig3b_{RUN_NAME}_substitutions_mapping.svg",
    show_figure=True
)

In [ ]:
df_mapped = visualization.create_dataframe_from_mapped_sequences(data=mapped_scaffolds)

In [ ]:
protein_norm